In [ ]:
import requests
import pandas as pd
import time

# Función principal: extrae todos los conciertos de Madrid para un año dado
def obtener_conciertos_por_año(api_key, year):
    url = "https://api.setlist.fm/rest/1.0/search/setlists"
    headers = {
        "Accept": "application/json",
        "x-api-key": api_key
    }
    # Parámetros de búsqueda: ciudad, país, año y primera página
    params = {
        "cityName": "Madrid",
        "countryCode": "ES",
        "year": year,
        "p": 1
    }
    resultados = []

    # Bucle de paginación: recorre todas las páginas de resultados
    while True:
        print(f"Año {year} - Página {params['p']}")
        r = requests.get(url, headers=headers, params=params)

        # Gestión del rate limit: espera 60s si la API devuelve error 429
        if r.status_code == 429:
            print("Rate limit. Esperando 60s...")
            time.sleep(60)
            continue

        # Cualquier otro error detiene la extracción
        if r.status_code != 200:
            print(f"Error {r.status_code}: {r.text}")
            break

        data = r.json()
        setlists = data.get("setlist", [])

        # La API a veces devuelve un dict en lugar de lista si hay un solo resultado
        if isinstance(setlists, dict):
            setlists = [setlists]

        # Si no hay resultados, se ha llegado al final
        if not setlists:
            break

        for sl in setlists:
            artist = sl.get("artist", {})
            venue  = sl.get("venue", {})
            city   = venue.get("city", {})
            sets   = sl.get("sets", {}).get("set", [])

            # Filtro: solo conciertos celebrados en Madrid, España
            if city.get("name") != "Madrid" or city.get("country", {}).get("code") != "ES":
                continue

            # Número total de canciones tocadas en el concierto
            n_canciones = sum(len(s.get("song", [])) for s in sets)
            # Indicador de si hubo bis (encore)
            hay_encore  = any(s.get("encore") for s in sets)

            resultados.append({
                # Evento
                "id_evento":     sl.get("id"),
                "fecha":         sl.get("eventDate"),
                "last_updated":  sl.get("lastUpdated"),
                "info_evento":   sl.get("info"),
                "url":           sl.get("url"),
                # Artista
                "artista":       artist.get("name"),
                "mbid":          artist.get("mbid"),        # ID MusicBrainz, clave para cruzar con Spotify
                "artista_alias": artist.get("disambiguation"),
                # Recinto
                "recinto":       venue.get("name"),
                "recinto_id":    venue.get("id"),
                # Ciudad (para validar en EDA)
                "ciudad":        city.get("name"),
                "pais":          city.get("country", {}).get("code"),
                # Gira
                "tour":          sl.get("tour", {}).get("name") if sl.get("tour") else None,
                # Setlist
                "n_canciones":   n_canciones,
                "hay_encore":    hay_encore,
            })

        total    = data.get("total", 0)
        per_page = data.get("itemsPerPage", 20)
        print(f"  -> Total este año: {total}")

        # Condición de parada: se han recorrido todas las páginas
        if params["p"] * per_page >= total:
            break
        params["p"] += 1
        time.sleep(2)  # Pausa entre peticiones para evitar el rate limit

    return pd.DataFrame(resultados)


# --- Ejecución ---
#MI_API_KEY = "..."  # Clave de acceso a la API de setlist.fm

años = [2022, 2023, 2024, 2025, 2026]
dfs = []

# Descarga año a año y guarda un CSV por año como copia de seguridad
for year in años:
    df_year = obtener_conciertos_por_año(MI_API_KEY, year)
    df_year.to_csv(f"conciertos_madrid_{year}.csv", index=False)
    print(f"{year}: {len(df_year)} filas guardadas\n")
    dfs.append(df_year)
    time.sleep(5)  # Pausa entre años

# Concatenación de todos los años en un único dataset
df_total = pd.concat(dfs, ignore_index=True)
df_total.to_csv("conciertos_madrid_2022_2026.csv", index=False)
print(f"Dataset completo: {len(df_total)} filas | {df_total['fecha'].min()} -> {df_total['fecha'].max()}")

In [ ]:
# --- Verificación del dataset ---
print("=== VERIFICACION DATASET SETLIST.FM ===\n")
print(f"Total filas: {len(df_total)}")
print(f"\nFilas por año:")
df_total['año'] = pd.to_datetime(df_total['fecha'], format='%d-%m-%Y').dt.year
print(df_total['año'].value_counts().sort_index())
print(f"\nColumnas: {list(df_total.columns)}")
print(f"\nValores nulos por columna:")
print(df_total.isnull().sum())
print(f"\nValores unicos en 'ciudad': {df_total['ciudad'].unique()}")
print(f"Valores unicos en 'pais': {df_total['pais'].unique()}")
print(f"\nPrimeras filas:")
df_total.head()